## Rating Prediction
 - Use Suprise SVD library (latent factor)
 - Use grid search for parameter tuning reg_bu, reg_bi, reg_pu, reg_qi, lr_bu, lr_bi, lr_pu, lr_qi
 - Test Review Time impact to actual result
    
   

In [1]:
# pip install surprise

In [2]:
# import library
import numpy as np
import pandas as pd
import os
import json
import ast
from datetime import datetime

from surprise import Reader, Dataset, SVD, SVDpp,KNNWithZScore
from scipy import sparse
from scipy.sparse import csr_matrix
from sklearn.utils.extmath import randomized_svd
from sklearn.metrics import mean_squared_error
from scipy import sparse

from sklearn.model_selection import train_test_split

In [3]:
# import file
with open('train.json') as f:
    raw=[ast.literal_eval(l) for l in f.readlines()]
df = pd.DataFrame(raw)

In [4]:
# used to test review time impact
df=df.sort_values(ascending=True,by=['unixReviewTime'])
df2 = df[['reviewerID','itemID', 'rating']]

_80_percent_mark = int(0.8*df2.shape[0])
train_df = df2[0:_80_percent_mark][['reviewerID','itemID', 'rating']]
test_df = df2[_80_percent_mark:][['reviewerID','itemID', 'rating']]

#train_df, test_df = train_test_split(df2, test_size=0.2 )

In [5]:
# Prepare data for surprise library
reader = Reader(rating_scale=(1,5))
train_data = Dataset.load_from_df(train_df[['reviewerID','itemID', 'rating']], reader)
trainset = train_data.build_full_trainset() 
testset = list(zip(test_df.reviewerID.values, test_df.itemID.values, test_df.rating.values))


### Define Train Test RMSE function

In [6]:
def rmse(predict):
    pred=[]
    actual=[]
    for p in predict:
        actual.append(p.r_ui)
        pred.append(p.est)
    actual=np.array(actual)
    pred=np.array(pred)
    rmse=np.sqrt(np.mean( (pred-actual)**2 ))
    return actual, pred, rmse

In [7]:
def get_result(model, trainset):
    model.fit(trainset)
    train_pred = model.test(trainset.build_testset())
    train_actual_ratings, train_pred_ratings, train_rmse = rmse(train_pred)
    test_pred = model.test(testset)
    test_actual_ratings, test_pred_ratings, test_rmse = rmse(test_pred)
    print('Train RMSE: {}; Test RMSE: {}'.format(train_rmse,test_rmse))

#### Test run

In [8]:
svd1 = SVD(n_factors=1, verbose=False,n_epochs=30,reg_bu=0.001, 
         lr_all=0.005, reg_all=0.4)
get_result(svd1,trainset = trainset)


Train RMSE: 0.8818172294758018; Test RMSE: 1.096836527697627


### Tune Parameter

In [9]:
from surprise.model_selection import GridSearchCV
# smaller grid for testing
param_grid = {'n_epochs':[30,30],
              "lr_all": [0.002,0.005],
              'n_factors':[1, 1],
              'reg_bu':[0.001,0.002],
              'reg_bi':[0.001,0.002],
              'reg_all':[0.5,0.6]}
 
grid_search = GridSearchCV(SVD, param_grid, measures=['RMSE'])
grid_search.fit(train_data)
training_parameters = grid_search.best_params["rmse"]
print("BEST RMSE: \t", grid_search.best_score["rmse"])
print("BEST params: \t", grid_search.best_params["rmse"])

BEST RMSE: 	 1.0579459732068746
BEST params: 	 {'n_epochs': 30, 'lr_all': 0.005, 'n_factors': 1, 'reg_bu': 0.001, 'reg_bi': 0.002, 'reg_all': 0.5}


In [10]:
svd2 = SVD(n_factors=1, verbose=False,n_epochs=30,reg_bu=0.001, 
         lr_all=0.005, reg_all=0.5)
get_result(svd2,trainset = trainset)


Train RMSE: 0.8839179024482477; Test RMSE: 1.0966460130838784


In [11]:
svd3 = SVD(n_factors=1, verbose=False,n_epochs=30,reg_bu=0.0005, reg_bi=0.002,reg_all=0.6,
         lr_bu=0.006,lr_bi=0.006,lr_all=0.001)
get_result(svd3,trainset = trainset)


Train RMSE: 0.8598247138703246; Test RMSE: 1.1019971052522142


In [12]:
svd4 = SVD(n_factors=1, verbose=False,n_epochs=30,reg_bu=0.001, reg_bi=0.001,reg_all=0.6,
         lr_bu=0.001,lr_bi=0.001,lr_all=0.005)
get_result(svd4,trainset = trainset)


Train RMSE: 1.010550360260217; Test RMSE: 1.1085629681716858


In [13]:
svd5 = SVD(n_factors=1, verbose=False,n_epochs=30,reg_bu=0.001, 
         lr_all=0.005, reg_all=0.6)
get_result(svd5,trainset = trainset)


Train RMSE: 0.8859796082599549; Test RMSE: 1.0965573916941957


In [14]:
svd6 = SVD(n_factors=1, verbose=False,n_epochs=30,reg_bu=0.002, 
         lr_all=0.005, reg_all=0.5)
get_result(svd6,trainset = trainset)


Train RMSE: 0.8839482835239061; Test RMSE: 1.09662795393893


#### Best Result  svd7

In [15]:
### BEST SO FAR ### 
svd7 = SVD(n_factors=1, verbose=False,n_epochs=30,reg_bu=0.002, 
         lr_all=0.005, reg_all=0.6)
get_result(svd7,trainset = trainset)


Train RMSE: 0.8859899909416984; Test RMSE: 1.096557109411191


In [16]:
svd8 = SVD(n_factors=1, verbose=False,n_epochs=30,#reg_bu=0.002, 
         lr_all=0.005, reg_all=0.6)
get_result(svd8,trainset = trainset)


Train RMSE: 0.9035169426778404; Test RMSE: 1.0963189116587597


### Generate final result


In [18]:
# train with full dataset
full_train_data = Dataset.load_from_df(df2[['reviewerID','itemID', 'rating']], reader)
fulltrainset = full_train_data.build_full_trainset() 
svd7.fit(fulltrainset)
train_preds = svd7.test(fulltrainset.build_testset())
train_actual_ratings, train_pred_ratings, train_rmse = rmse(train_preds)
print("Train rmse : {} ".format(train_rmse))  


Train rmse : 0.8948112432703074 


In [19]:
predictions_ratings = open("predictions_Rating.csv", 'w')

for l in open("pairs_Rating.txt"):
    if l.startswith("reviewerID"):
        predictions_ratings.write(l)
        continue
    u,i = l.strip().split('-')
    predictions_ratings.write(u + '-' + i + ',' + str(svd7.predict(u,i).est) + '\n')
predictions_ratings.close()